# Tutorial: Phase 1 ViewState + Axis Selectors (Rust Backend)

Audience:
- Engineers validating end-to-end ViewState behavior through the Rust daemon.

Prerequisites:
- Run `uv sync` in this repository.
- Rust toolchain and `cargo` are available locally.

Learning goals:
- Start a real Rust daemon instance.
- Open a dataset through the Rust HTTP API.
- Mutate ViewState selectors and verify normalization, hash updates, and version increments.


## Step 1 - Imports, repository resolution, and OME-Zarr fixture helper


In [1]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import zarr

from lucida.client import LucidaClient, LucidaClientError

cwd = Path.cwd().resolve()
if (cwd / 'MIGRATION.md').exists():
    REPO_ROOT = cwd
elif (cwd.parent.parent / 'MIGRATION.md').exists():
    REPO_ROOT = cwd.parent.parent
else:
    raise AssertionError('Could not locate repository root containing MIGRATION.md')

sys.path.insert(0, str(REPO_ROOT / 'tests'))
from rust_daemon import start_rust_daemon  # noqa: E402


def build_demo_omezarr(dataset_path: Path) -> str:
    dataset_path.mkdir(parents=True, exist_ok=True)
    root = zarr.open_group(store=str(dataset_path), mode='w')

    shape_0 = (1, 2, 4, 8, 10)
    shape_1 = (1, 2, 2, 4, 5)
    root.create_array(
        '0',
        data=np.arange(np.prod(shape_0), dtype=np.uint16).reshape(shape_0),
        chunks=(1, 1, 2, 4, 5),
        overwrite=True,
    )
    root.create_array(
        '1',
        data=np.arange(np.prod(shape_1), dtype=np.uint16).reshape(shape_1),
        chunks=(1, 1, 1, 2, 3),
        overwrite=True,
    )

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z', 'unit': 'micron'},
                {'name': 'y', 'type': 'y', 'unit': 'micron'},
                {'name': 'x', 'type': 'x', 'unit': 'micron'},
            ],
            'datasets': [
                {
                    'path': '0',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 1, 1, 1]},
                        {'type': 'translation', 'translation': [0, 0, 0, 0, 0]},
                    ],
                },
                {
                    'path': '1',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 2, 2, 2]}
                    ],
                },
            ],
        }
    ]
    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'DNA', 'color': 'FF0000', 'window': {'start': 10, 'end': 400}},
            {'index': 1, 'label': 'RNA', 'color': '00FF00', 'window': {'start': 20, 'end': 200}},
        ]
    }

    return str(dataset_path)

print(f'REPO_ROOT={REPO_ROOT}')


REPO_ROOT=/Users/austin/GitHub/lucida


## Step 2 - Build dataset and start the Rust daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-viewstate-rust-'))
dataset_uri = build_demo_omezarr(tmp_dir / 'viewstate-demo.zarr')

rust_daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=rust_daemon.base_url)

print('dataset_uri:', dataset_uri)
print('rust_base_url:', rust_daemon.base_url)


dataset_uri: /var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-viewstate-rust-gpu23mp1/viewstate-demo.zarr
rust_base_url: http://127.0.0.1:50948


## Step 3 - Create session, open dataset, and create a view through Rust


In [3]:
session = client.create_session()
opened = client.open_dataset(uri=dataset_uri, session_id=session.session_id)
created = client.create_view(
    dataset_id=opened.dataset_summary.dataset_id,
    session_id=session.session_id,
    mode='2d',
)

print('session_id:', session.session_id)
print('dataset_id:', opened.dataset_summary.dataset_id)
print('view_id:', created.view_state.view_id)
print('initial state_version:', created.view_state.state_version)
print('initial state_hash:', created.view_state.state_hash)


session_id: session_c85ce8c0fde44e7a
dataset_id: ds_c461c96a3ac2d737
view_id: view_e108214af8de4474
initial state_version: 0
initial state_hash: 05f485efe7d016ec0e4eb92637dccd261a178a741dc68fe72f5024040bd6782e


## Step 4 - Apply selector helpers and validate invariants


In [4]:
updated_index = client.set_dim(
    view_id=created.view_state.view_id,
    axis='z',
    index=2,
    session_id=session.session_id,
)
updated_range = client.set_axis_range(
    view_id=created.view_state.view_id,
    axis='z',
    start=1,
    end_exclusive=4,
    session_id=session.session_id,
)
updated_set = client.set_axis_set(
    view_id=created.view_state.view_id,
    axis='z',
    indices=[0, 2, 2, 3],
    session_id=session.session_id,
)

z_index_selector = next(s for s in updated_index.selectors_applied if s.axis == 'z')
z_range_selector = next(s for s in updated_range.selectors_applied if s.axis == 'z')
z_set_selector = next(s for s in updated_set.selectors_applied if s.axis == 'z')

assert z_index_selector.index == 2
assert z_range_selector.start == 1 and z_range_selector.end_exclusive == 4
assert z_set_selector.indices == [0, 2, 3]

assert created.view_state.state_version == 0
assert updated_index.view_state.state_version == 1
assert updated_range.view_state.state_version == 2
assert updated_set.view_state.state_version == 3

assert created.view_state.state_hash != updated_index.view_state.state_hash
assert updated_index.view_state.state_hash != updated_range.view_state.state_hash
assert updated_range.view_state.state_hash != updated_set.view_state.state_hash

print('Selector and version/hash assertions passed against Rust backend.')


Selector and version/hash assertions passed against Rust backend.


## Step 5 - Validate strict selector behavior via Rust errors


In [5]:
try:
    client.update_view(
        view_id=created.view_state.view_id,
        session_id=session.session_id,
        patch=[
            {
                'op': 'replace',
                'path': '/selectors',
                'value': [{'axis': 'z', 'kind': 'index', 'index': 999, 'clamp': False}],
            }
        ],
    )
    raise AssertionError('Expected strict selector update to fail')
except LucidaClientError as exc:
    message = str(exc)
    assert 'selector_out_of_bounds' in message
    print('Observed expected strict-selector error from Rust backend.')


Observed expected strict-selector error from Rust backend.


## Step 6 - Retrieve view payload from Rust and inspect final state


In [6]:
fetched = client.get_view(view_id=created.view_state.view_id, session_id=session.session_id)
assert fetched.view_state.view_id == created.view_state.view_id
assert fetched.view_state.session_id == session.session_id
assert fetched.view_state.state_version == 3

preview = fetched.view_state.model_dump(mode='json')
print(json.dumps({
    'view_id': preview['view_id'],
    'session_id': preview['session_id'],
    'state_version': preview['state_version'],
    'state_hash': preview['state_hash'],
    'selectors': preview['selectors'],
}, indent=2))


{
  "view_id": "view_e108214af8de4474",
  "session_id": "session_c85ce8c0fde44e7a",
  "state_version": 3,
  "state_hash": "47c8fae78d20681dc60e22463750cf377c3b3ff6784443d4c32a151494e3adbb",
  "selectors": [
    {
      "axis": "t",
      "kind": "index",
      "index": 0,
      "start": null,
      "end_exclusive": null,
      "indices": null,
      "clamp": true
    },
    {
      "axis": "c",
      "kind": "index",
      "index": 0,
      "start": null,
      "end_exclusive": null,
      "indices": null,
      "clamp": true
    },
    {
      "axis": "z",
      "kind": "set",
      "index": null,
      "start": null,
      "end_exclusive": null,
      "indices": [
        0,
        2,
        3
      ],
      "clamp": true
    }
  ]
}


## Step 7 - Cleanup


In [7]:
client.close()
rust_daemon.stop()
print('Rust daemon stopped.')


Rust daemon stopped.


## Expected output checks

After running all cells, verify:
- A non-empty Rust daemon base URL is printed.
- Non-empty `session_id`, `dataset_id`, and `view_id` are printed.
- `Selector and version/hash assertions passed against Rust backend.` is printed.
- The strict selector update emits an expected `selector_out_of_bounds` error.
- Final `state_version` equals `3` and final `z` selector is `[0, 2, 3]`.
